## Import necessary libraries

In [116]:
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import os

Imports - All required libraries
load_stl() - Loads STL file and returns mesh object
find_undercuts() - Finds upward-facing surfaces that can't be reached from above
narrow slots
steep walls
deep pockets
small surfaces
internal cavities


In [117]:
def load_stl(stl_path):
    try:
        mesh = trimesh.load(stl_path)
        print(f"Loaded STL file successfully: {stl_path}")
        print(f"faces: {len(mesh.faces)}, vertices: {len(mesh.vertices)}")
        return mesh
    except Exception as e:
        print(f"Failed to load: {e}")
        return None


In [126]:
cube = load_stl(r'C:\Users\junhongs\Desktop\itp\ITP-CAD\test_file\simple\cube.stl')


Loaded STL file successfully: C:\Users\junhongs\Desktop\itp\ITP-CAD\test_file\simple\cube.stl
faces: 12, vertices: 8


In [131]:
bracket = load_stl(r'C:\Users\junhongs\Desktop\itp\ITP-CAD\test_file\moderate\38mm-90degreeBracket.STL')

Loaded STL file successfully: C:\Users\junhongs\Desktop\itp\ITP-CAD\test_file\moderate\38mm-90degreeBracket.STL
faces: 348, vertices: 172


Find overhangs - Downward-facing surfaces

In [129]:
# change the direction of model that is being loaded, to ensure that undercuts are detected correctly, if change of model orientation can pass the undercut test, it can still be machines by cnc

# if this orientation passes this test, continue the rest of the test with this orientation.

# find the necessary test cases for each of the tests, then run the whole script.








def find_undercuts(mesh):
    """Simplified and working version of find_undercuts"""
    face_normals = mesh.face_normals
    face_centers = mesh.triangles_center
    mesh_center = np.mean(mesh.vertices, axis=0)

    undercut_faces = []

    for i, (center, normal) in enumerate(zip(face_centers, face_normals)):
        # Look for faces that point upward (positive Z)
        if normal[2] > 0.3:  # Face points somewhat upward

            # Check if this face is "hidden" inside the mesh
            # by comparing face direction with direction from mesh center
            to_face = center - mesh_center
            to_face_norm = to_face / (np.linalg.norm(to_face) + 1e-8)

            # If face normal and position vector point in opposite directions,
            # this suggests an internal/undercut surface
            alignment = np.dot(normal, to_face_norm)

            if alignment < -0.2:  # Face points inward relative to its position
                undercut_faces.append(i)

    undercut_indices = np.array(undercut_faces)
    print(f"Found {len(undercut_indices)} potential undercut faces")
    return undercut_indices


In [132]:
braket_undercuts = find_undercuts(bracket)

Found 28 potential undercut faces


In [133]:
cube_undercuts = find_undercuts(cube) # control test

Found 0 potential undercut faces


In [124]:
def find_internal_volumes(mesh):

# Check if watertight (required for internal volume detection)
    print(f"Is watertight: {mesh.is_watertight}")
    if not mesh.is_watertight:
        print("Mesh is not watertight - cannot detect internal volumes reliably")
        # meaning that its not enclosed if its not watertight
        print("STL files that have gaps or holes")
        return

    # Calculate volumes
    try:
        actual_volume = mesh.volume
        # convex hull is a property from trimesh
        # it calculates the smallest "shrink-wrap" that can completely enclose the 3D object without any inward curves.

        convex_volume = mesh.convex_hull.volume

        print(f"Actual volume: {actual_volume:.3f}")
        print(f"Convex hull volume: {convex_volume:.3f}")

        # if convex hull volume == 1, its solid object
        # if convex hull volume == 0, its hollow object
        # this show interal cavities
        if actual_volume > 0 and convex_volume > 0:
            volume_ratio = actual_volume / convex_volume
            print(f"Volume ratio (actual/convex): {volume_ratio:.3f}")

            print(f"\nInterpretation:")
            if volume_ratio > 0.9:
                print(f"✓ Solid object (ratio > 0.9)")
            elif volume_ratio > 0.6:
                print(f"⚠ Minor internal spaces (ratio 0.6-0.9)")
            else:
                print(f"🕳 Significant internal volumes (ratio < 0.6)")
                print(f"   This suggests the object is hollow inside!")

        else:
            print("Invalid volume calculations")

    except Exception as e:
        print(f"Error calculating volumes: {e}")

In [86]:
cone = load_stl(r'C:\Users\junhongs\Desktop\itp\ITP-CAD\test_file\simple\cone.stl')

Loaded STL file successfully: C:\Users\junhongs\Desktop\itp\ITP-CAD\test_file\simple\cone.stl
faces: 41000, vertices: 20502


In [91]:
internal_vols = find_internal_volumes(vase)


Is watertight: True
Actual volume: 249209.606
Convex hull volume: 1468463.389
Volume ratio (actual/convex): 0.170

Interpretation:
🕳 Significant internal volumes (ratio < 0.6)
   This suggests the object is hollow inside!


In [94]:
internal_vols = find_internal_volumes(cone)

Is watertight: False
Mesh is not watertight - cannot detect internal volumes reliably
STL files that have gaps or holes


In [109]:
interal_vol_cube = find_internal_volumes(cube) # control test

Is watertight: True
Actual volume: 8000.000
Convex hull volume: 8000.000
Volume ratio (actual/convex): 1.000

Interpretation:
✓ Solid object (ratio > 0.9)


In [113]:
def find_deep_pockets_enhanced(mesh, depth_threshold=30.0):
    """
    Find faces in very deep pockets where tool length becomes an issue.
    Depth-to-width ratio > 5:1 is typically problematic.
    """
    face_centers = mesh.triangles_center
    face_normals = mesh.face_normals

    deep_faces = []

    print(f"Checking for pockets deeper than {depth_threshold} units...")

    for i, (center, normal) in enumerate(zip(face_centers, face_normals)):
        # Estimate depth by measuring distance to part boundary
        depth = estimate_pocket_depth(mesh, center, normal)

        if depth > depth_threshold:
            deep_faces.append(i)

    print(f"Found {len(deep_faces)} faces in deep pockets")
    return np.array(deep_faces)

# Test function:
deep_pockets_enhanced = find_deep_pockets_enhanced(vase)

Checking for pockets deeper than 30.0 units...
Found 45980 faces in deep pockets


In [103]:
def estimate_pocket_depth(mesh, center, normal):
    """Estimate the depth of a pocket at a specific face."""
    try:
        # Cast ray outward from face
        locations, _, _ = mesh.ray.intersects_location(
            ray_origins=center.reshape(1, -1),
            ray_directions=normal.reshape(1, -1)
        )

        if len(locations) > 0:
            depths = np.linalg.norm(locations - center, axis=1)
            valid_depths = depths[depths > 0.1]  # Ignore very close hits
            return np.min(valid_depths) if len(valid_depths) > 0 else 0

        return 0
    except:
        return 0



Steep walls - Nearly vertical faces

In [42]:
def find_steep_walls(mesh):
    face_normals = mesh.face_normals

    # Find vertical walls (these are actually EASY for CNC)
    vertical_mask = np.abs(face_normals[:, 2]) < 0.2

    # For CNC, ignore the straight walls - only flag if in problematic locations
    # For now, let's just return a reasonable subset
    steep_indices = np.where(vertical_mask)[0]

    print(f"Found {len(steep_indices)} vertical wall faces")
    return steep_indices

In [100]:
steep_walls = find_steep_walls(vase)

Found 25411 vertical wall faces


Deep pockets - Concave regions

In [45]:
def find_deep_pockets(mesh):
    """Find faces that might be in deep pockets using normal analysis."""
    face_normals = mesh.face_normals
    face_centers = mesh.triangles_center
    mesh_center = np.mean(mesh.vertices, axis=0)

    pocket_faces = []
    for i, (center, normal) in enumerate(zip(face_centers, face_normals)):
        to_face = center - mesh_center
        to_face_norm = to_face / (np.linalg.norm(to_face) + 1e-8)
        if np.dot(normal, -to_face_norm) > 0.3:
            pocket_faces.append(i)

    pocket_indices = np.array(pocket_faces)
    print(f"Found {len(pocket_indices)} deep pocket faces")
    return pocket_indices

In [47]:
vase = load_stl(r'C:\Users\junhongs\Desktop\itp\ITP-CAD\test_file\vase.stl')

Loaded STL file successfully: C:\Users\junhongs\Desktop\itp\ITP-CAD\test_file\vase.stl
faces: 150306, vertices: 75155


In [48]:
deep_pockets = find_deep_pockets(vase)

Found 47183 deep pocket faces


In [59]:
def find_small_features(mesh, min_tool_diameter=3.0, min_feature_size=1.0):
    """Find features too small for standard CNC tools."""

    # Get edge lengths in actual units (mm)
    edge_lengths = mesh.edges_unique_length

    # Small features: edges smaller than minimum tool radius
    very_small_edges = edge_lengths < (min_tool_diameter / 2)
    small_edges = edge_lengths < min_feature_size

    # Calculate percentages
    very_small_pct = len(very_small_edges[very_small_edges]) / len(edge_lengths) * 100
    small_pct = len(small_edges[small_edges]) / len(edge_lengths) * 100

    # Flag as having small features if significant portion is small
    has_small_features = very_small_pct > 5 or small_pct > 15

    print(f"Edge length stats:")
    print(f"  Min: {np.min(edge_lengths):.2f}mm")
    print(f"  Max: {np.max(edge_lengths):.2f}mm")
    print(f"  Mean: {np.mean(edge_lengths):.2f}mm")
    print(f"  Very small edges (<{min_tool_diameter/2}mm): {very_small_pct:.1f}%")
    print(f"  Small edges (<{min_feature_size}mm): {small_pct:.1f}%")
    print(f"Has small features: {has_small_features}")

    return has_small_features

In [60]:
small_features = find_small_features(bracket)

Edge length stats:
  Min: 0.34mm
  Max: 52.00mm
  Mean: 9.13mm
  Very small edges (<1.5mm): 28.4%
  Small edges (<1.0mm): 28.4%
Has small features: True


In [61]:
small_features = find_small_features(cube)

Edge length stats:
  Min: 20.00mm
  Max: 28.28mm
  Mean: 22.76mm
  Very small edges (<1.5mm): 0.0%
  Small edges (<1.0mm): 0.0%
Has small features: False


In [62]:
small_features = find_small_features(vase)

Edge length stats:
  Min: 0.01mm
  Max: 45.00mm
  Mean: 1.84mm
  Very small edges (<1.5mm): 39.0%
  Small edges (<1.0mm): 26.8%
Has small features: True


In [ ]:
def calculate_cnc_manufacturability_score(mesh):
    """
    Analyze manufacturability for CNC milling.
    Returns score 0-100 (100 = perfect for CNC, 0 = impossible)
    """
    manufacturability_score = 100
    problem_regions = []

    print("Analyzing CNC manufacturability...")

    # 1. Check for undercuts (biggest CNC problem)
    undercuts = find_undercuts(mesh)
    if len(undercuts) > 0:
        penalty = min(50, len(undercuts) * 0.1)  # Up to 50 point penalty
        manufacturability_score -= penalty
        problem_regions.append(("Undercuts (Cannot machine)", undercuts))

    # 2. Check for enclosed internal volumes (impossible to machine)
    internal_volumes = find_internal_volumes(mesh)
    if internal_volumes > 0:
        manufacturability_score -= 40
        problem_regions.append(("Internal Volumes", []))

    # 3. Check for very narrow channels (tool access limited)
    narrow_channels = find_narrow_channels(mesh)
    if len(narrow_channels) > 0:
        penalty = min(25, len(narrow_channels) * 0.05)
        manufacturability_score -= penalty
        problem_regions.append(("Narrow Channels", narrow_channels))

    # 4. Check for very deep pockets (tool length/rigidity issues)
    deep_pockets = find_deep_pockets_enhanced(mesh)
    if len(deep_pockets) > 0:
        penalty = min(20, len(deep_pockets) * 0.03)
        manufacturability_score -= penalty
        problem_regions.append(("Deep Pockets", deep_pockets))

    # 5. Check for small features (below minimum tool size)
    if find_small_features(mesh):
        manufacturability_score -= 15
        problem_regions.append(("Small Features", []))

    manufacturability_score = max(0, manufacturability_score)
    print(f"Final CNC Manufacturability Score: {manufacturability_score}/100")

    return manufacturability_score, problem_regions

In [ ]:
def plot_cnc_analysis(mesh, problem_regions, manufacturability_score):
    """Plot mesh with CNC problem regions highlighted."""
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    vertices = mesh.vertices
    faces = mesh.faces

    # Default color for all faces (good for CNC)
    face_colors = ['lightgreen'] * len(faces)  # Green = good for CNC

    # Color problem regions
    colors = ['red', 'orange', 'yellow', 'purple']
    for i, (region_name, face_indices) in enumerate(problem_regions):
        color = colors[i % len(colors)]
        for face_idx in face_indices:
            if face_idx < len(face_colors):
                face_colors[face_idx] = color

    # Create colored mesh
    mesh_3d = Poly3DCollection(vertices[faces], alpha=0.8,
                               facecolors=face_colors, edgecolor='black', linewidth=0.1)
    ax.add_collection3d(mesh_3d)

    # Set axis limits
    ax.set_xlim(vertices[:, 0].min(), vertices[:, 0].max())
    ax.set_ylim(vertices[:, 1].min(), vertices[:, 1].max())
    ax.set_zlim(vertices[:, 2].min(), vertices[:, 2].max())

    ax.set_title(f'CNC Manufacturability Analysis\nScore: {manufacturability_score}/100',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    # Add legend
    legend_text = ["Green: Good for CNC"]
    if problem_regions:
        for i, (region_name, _) in enumerate(problem_regions):
            legend_text.append(f"{colors[i % len(colors)]}: {region_name}")

    ax.text2D(0.02, 0.98, '\n'.join(legend_text), transform=ax.transAxes,
              verticalalignment='top', fontsize=9,
              bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    plt.show()
    return fig

# Test this function:
# plot_cnc_analysis(mesh, cnc_regions, cnc_score)

Models with: (Should show RED regions)
- Horizontal ledges
- T-sections
- Cantilevers
- Bridge structures
- 
Models with:
(Should show ORANGE regions)
- Tall thin walls
- Vertical ribs
- Deep slots
- Tower-like features

Models with: (Should show YELLOW regions)
- Internal cavities
- Deep holes
- Recessed features
- Mold-like shapes

2. Models with UNDERCUTS (Should Score LOW - RED regions)
Snap-Fit Features

"Snap fit connector" - Plastic parts with flexible tabs
"Clip" / "Clasp" - Parts that clip onto other parts
"Phone case" - Often has undercuts around edges

Dovetail Features

"Dovetail joint" - Woodworking joints with angled cuts
"T-slot" / "T-groove" - Slots that widen internally
"Keyway" - Shaft keyways that are wider at bottom

Threaded Features

"Threaded rod" - External threads (moderate undercut)
"Nut" - Internal threads (severe undercut)
"Screw" - Thread profiles

3. Models with INTERNAL VOLUMES (Should Score LOW)
Hollow Objects

"Hollow sphere" - Completely enclosed internal space
"Bottle" / "Vase" - Internal cavity accessible only through small opening
"Container with lid" - If modeled as one piece

Complex Internal Geometry

"Manifold" - Internal passages and chambers
"Valve body" - Internal flow passages
"Engine block" - Internal coolant/oil passages
"Heat exchanger" - Internal channels

4. Models with NARROW CHANNELS (Should Score MEDIUM - ORANGE)
Thin Slot Features

"Thin wall part" - Parts with very thin walls
"Slot" / "Groove" - Narrow channels
"Heat sink" - Narrow fins and channels
"Ribbed part" - Parts with many thin ribs

Tight Clearances

"Close tolerance part" - Parts requiring precision
"Sliding mechanism" - Parts with tight fits

5. Models with DEEP POCKETS (Should Score MEDIUM - YELLOW)
Deep Cavities

"Mold cavity" - Deep forming molds
"Deep pocket part" - Parts with recessed areas
"Tool holder" - Deep holes for holding tools
"Socket" - Deep hexagonal or square holes

High Aspect Ratio Features

"Deep hole part" - Parts with very deep holes
"Tall tower" - Thin, tall features
"Deep recess" - Recessed mounting areas

"manifold" (internal volumes)
"snap fit" (undercuts)
"heat sink" (narrow channels)
"mold" (deep pockets)
"valve" (complex internal)
"bracket" (CNC-friendly baseline)